<a href="https://colab.research.google.com/github/davidewetuga/NBA_MultiYear_Prediction_py.ipynb/blob/main/NBA_MultiYear_Prediction_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
NBA Full Stat Sheet Prediction & Visualization
----------------------------------------------
1. Fetches Multi-Year History (2021-Present)
2. Predicts PTS, REB, AST, 3PM, PRA for next game
3. Generates and saves a TREND GRAPH for each stat
"""

import pandas as pd
import numpy as np
import xgboost as xgb
import time
from nba_api.stats.endpoints import playergamelog
from nba_api.stats.static import players
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

# ===================================================
# 📌 CONFIGURATION
# ===================================================
PLAYER_NAME = 'Alex Caruso'
SEASONS_TO_ANALYZE = ['2021-22', '2022-23', '2023-24', '2024-25']
TARGETS = ['PTS', 'REB', 'AST', 'FG3M', 'PRA']
# ===================================================

def get_player_id(name):
    nba_players = players.find_players_by_full_name(name)
    if not nba_players:
        raise ValueError(f"Player '{name}' not found.")
    return nba_players[0]['id']

def fetch_multi_season_logs(player_name, seasons):
    pid = get_player_id(player_name)
    all_season_dfs = []

    print(f"--- Fetching History for {player_name} ---")
    for season in seasons:
        try:
            print(f"Downloading {season}...")
            gl = playergamelog.PlayerGameLog(player_id=pid, season=season)
            df = gl.get_data_frames()[0]
            df['SEASON_ID'] = season
            if not df.empty:
                all_season_dfs.append(df)
            time.sleep(0.5)
        except Exception as e:
            print(f"Skipping {season}: {e}")

    if not all_season_dfs:
        raise ValueError("No data found.")

    full_df = pd.concat(all_season_dfs, ignore_index=True)
    full_df['GAME_DATE'] = pd.to_datetime(full_df['GAME_DATE'])
    full_df = full_df.sort_values('GAME_DATE').reset_index(drop=True)

    return full_df

def main():
    # A. Load Multi-Year Data
    df = fetch_multi_season_logs(PLAYER_NAME, SEASONS_TO_ANALYZE)

    # B. Data Prep & PRA Calculation
    cols_check = ['MIN', 'PTS', 'FG3M', 'REB', 'AST']
    for col in cols_check:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    df['PRA'] = df['PTS'] + df['REB'] + df['AST']

    # C. Feature Engineering
    print("Engineering features...")
    metrics_to_roll = ['PTS', 'MIN', 'FG3M', 'PRA', 'REB', 'AST']

    for col in metrics_to_roll:
        for window in [3, 7, 14, 30]:
            col_name = f"{col.lower()}_roll_{window}"
            df[col_name] = df[col].rolling(window=window).mean().shift(1)

    model_data = df.dropna().reset_index(drop=True)
    feature_cols = [c for c in model_data.columns if '_roll_' in c]
    X = model_data[feature_cols]

    print("\n" + "="*50)
    print(f"🤖 PREDICTIONS & GRAPHS FOR {PLAYER_NAME}")
    print("="*50)

    # --- LOOP THROUGH EACH TARGET ---
    for target in TARGETS:
        y = model_data[target].values

        # 1. Train Model
        model = xgb.XGBRegressor(objective='reg:absoluteerror', n_estimators=300, learning_rate=0.05, max_depth=5)
        model.fit(X, y)

        # 2. Predict Next Game
        last_games = df.tail(30)
        next_feats = {}
        for col in metrics_to_roll:
            for window in [3, 7, 14, 30]:
                feat_name = f"{col.lower()}_roll_{window}"
                next_feats[feat_name] = last_games[col].tail(window).mean()

        X_next = pd.DataFrame([next_feats])[feature_cols]
        prediction = model.predict(X_next)[0]

        # 3. Confidence Check (MAE on recent games)
        recent_preds = model.predict(X.tail(50))
        recent_mae = mean_absolute_error(y[-50:], recent_preds)

        print(f"🏀 {target}: {prediction:.2f}  (Confidence: +/- {recent_mae:.1f})")

        # ---------------------------------------------------------
        # 4. VISUALIZATION (Graph Generation)
        # ---------------------------------------------------------
        plt.figure(figsize=(14, 7))

        # Color code dots by Season so you can see history vs now
        seasons = model_data['SEASON_ID'].unique()
        colors = plt.cm.viridis(np.linspace(0, 1, len(seasons)))

        for i, season in enumerate(seasons):
            season_data = model_data[model_data['SEASON_ID'] == season]
            plt.scatter(season_data['GAME_DATE'], season_data[target],
                        label=season, color=colors[i], alpha=0.6, s=20)

        # Plot the "Long Term Trend" (30-game rolling average)
        trend_col = f"{target.lower()}_roll_30"
        plt.plot(model_data['GAME_DATE'], model_data[trend_col],
                 color='red', linewidth=2.5, label='Long-Term Trend')

        # Plot the prediction as a big gold star at the end
        last_date = model_data['GAME_DATE'].iloc[-1] + pd.Timedelta(days=2)
        plt.scatter(last_date, prediction, color='gold', marker='*', s=300,
                    edgecolor='black', label='Next Game Prediction', zorder=5)

        plt.title(f"{PLAYER_NAME}: {target} Trend Analysis & Prediction")
        plt.xlabel('Date')
        plt.ylabel(target)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()

        # Save unique file for each stat
        filename = f"{PLAYER_NAME}_{target}_Forecast.png"
        plt.savefig(filename)
        print(f"   └── Graph saved: {filename}")
        plt.close() # Close plot to save memory

    print("="*50)

if __name__ == "__main__":
    main()

--- Fetching History for Alex Caruso ---
Engineering features...

🤖 PREDICTIONS & GRAPHS FOR Alex Caruso
🏀 PTS: 10.15  (Confidence: +/- 0.9)
   └── Graph saved: Alex Caruso_PTS_Forecast.png
🏀 REB: 3.04  (Confidence: +/- 0.1)
   └── Graph saved: Alex Caruso_REB_Forecast.png
🏀 AST: 2.48  (Confidence: +/- 0.3)
   └── Graph saved: Alex Caruso_AST_Forecast.png
🏀 FG3M: 1.35  (Confidence: +/- 0.2)
   └── Graph saved: Alex Caruso_FG3M_Forecast.png
🏀 PRA: 15.06  (Confidence: +/- 1.2)
   └── Graph saved: Alex Caruso_PRA_Forecast.png
